# 06 – Tracking with Trackastra and evaluation with traccuracy (sequence 01)

**Purpose:** Link segmented nuclei over time (including divisions) and measure tracking quality against the
reference. The tracks are the basis for the motion analysis around mitosis (stage 3).

**Pipeline**
1. Segment all 92 frames of sequence 01 with the **fine-tuned** Cellpose-SAM model (notebook 04).
2. Track with **Trackastra** (pretrained `general_2d`, mode `greedy` = linking *with* divisions).
3. Evaluate with **traccuracy** against `01_GT/TRA` (CTC measures DET, LNK, TRA; division precision/recall/F1).
4. Comparison: the same tracking on the **silver-truth** masks – shows how much of the tracking error comes from segmentation.
5. Export one row per nucleus and frame for R.

**Tools:** trackastra 0.5.6, traccuracy 0.4.3 (installed into `bia`, 2026-11-03).

### Reference measures (CTC)
- **DET:** how well nuclei are detected per frame (0–1).
- **LNK:** how well consecutive detections are linked.
- **TRA:** combined tracking accuracy (detections + links + divisions), the main CTC tracking measure.
- **Division F1:** divisions found at the right time and with the right daughters; a buffer of ±k frames
  allows a division to be detected slightly early or late.

## 1. Settings

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from skimage.measure import regionprops_table

DATA = Path("../data/raw/Fluo-N2DL-HeLa")
DERIVED = Path("../data/derived")
RESULTS = Path("../results")
MODEL_FT = Path("../models/cpsam_hela_seq02")
SEQ = "01"
PIXEL_SIZE_UM = 0.645
FRAME_INTERVAL_MIN = 30

assert MODEL_FT.is_file(), f"fine-tuned model not found: {MODEL_FT.resolve()}"
DERIVED.mkdir(parents=True, exist_ok=True)

## 2. Load the raw time-lapse and the silver truth

In [2]:
raw = np.stack([tifffile.imread(f) for f in sorted((DATA / SEQ).glob("t*.tif"))])
st = np.stack([tifffile.imread(f) for f in sorted((DATA / f"{SEQ}_ST" / "SEG").glob("man_seg*.tif"))])
print("raw:", raw.shape, raw.dtype, "| silver truth:", st.shape, st.dtype)

raw: (92, 700, 1100) uint16 | silver truth: (92, 700, 1100) uint16


## 3. Segment all frames with the fine-tuned model (cached)
About 92 × 17–22 s ≈ 30 min on the M2. The label stack is saved to `data/derived/` and reloaded on later runs.

In [3]:
from cellpose import models

seg_file = DERIVED / f"masks_finetuned_seq{SEQ}.tif"
if seg_file.exists():
    seg = tifffile.imread(seg_file)
    print("loaded cached segmentation:", seg.shape)
else:
    model = models.CellposeModel(gpu=True, pretrained_model=str(MODEL_FT))
    seg = np.zeros(raw.shape, dtype=np.uint16)
    t0 = time.time()
    for t in range(len(raw)):
        seg[t] = model.eval(raw[t])[0]
        print(f"[{t + 1}/{len(raw)}] {seg[t].max()} nuclei, {time.time() - t0:.0f} s", flush=True)
    tifffile.imwrite(seg_file, seg, compression="zlib")
    print("saved:", seg_file)

/Users/inu/miniforge3/envs/bia/lib/python3.12/site-packages/cellpose/dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:923.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


[1/92] 41 nuclei, 19 s
[2/92] 41 nuclei, 36 s
[3/92] 40 nuclei, 53 s
[4/92] 44 nuclei, 70 s
[5/92] 44 nuclei, 88 s
[6/92] 44 nuclei, 106 s
[7/92] 45 nuclei, 122 s
[8/92] 47 nuclei, 139 s
[9/92] 49 nuclei, 156 s
[10/92] 50 nuclei, 173 s
[11/92] 52 nuclei, 190 s
[12/92] 53 nuclei, 207 s
[13/92] 52 nuclei, 226 s
[14/92] 55 nuclei, 246 s
[15/92] 54 nuclei, 268 s
[16/92] 58 nuclei, 290 s
[17/92] 58 nuclei, 311 s
[18/92] 58 nuclei, 332 s
[19/92] 58 nuclei, 352 s
[20/92] 59 nuclei, 373 s
[21/92] 59 nuclei, 393 s
[22/92] 59 nuclei, 414 s
[23/92] 60 nuclei, 434 s
[24/92] 64 nuclei, 455 s
[25/92] 66 nuclei, 477 s
[26/92] 66 nuclei, 498 s
[27/92] 66 nuclei, 519 s
[28/92] 66 nuclei, 540 s
[29/92] 65 nuclei, 561 s
[30/92] 67 nuclei, 583 s
[31/92] 67 nuclei, 604 s
[32/92] 69 nuclei, 625 s
[33/92] 70 nuclei, 646 s
[34/92] 68 nuclei, 668 s
[35/92] 72 nuclei, 689 s
[36/92] 72 nuclei, 711 s
[37/92] 72 nuclei, 733 s
[38/92] 71 nuclei, 754 s
[39/92] 75 nuclei, 775 s
[40/92] 78 nuclei, 797 s
[41/92] 81 nuc

## 4. Track with Trackastra
Trackastra predicts, for nuclei in nearby frames, how likely they belong to the same cell (a transformer looks at
position, shape and image features of all nuclei together). Mode `greedy` then links the most likely pairs and
allows one nucleus to link to two daughters (division).

`graph_to_ctc` relabels the masks so that each **track** has its own label and writes the result in CTC format
(label images + track table), the same format as the reference.

In [4]:
import torch
from trackastra.model import Trackastra
from trackastra.tracking import graph_to_ctc, graph_to_napari_tracks

device = "mps" if torch.backends.mps.is_available() else "cpu"
tracker = Trackastra.from_pretrained("general_2d", device=device)
print("Trackastra device:", device)

tracked = {}
for name, masks in [("finetuned", seg), ("silvertruth", st)]:
    t0 = time.time()
    graph, masks_tracked = tracker.track(raw, masks, mode="greedy")
    outdir = DERIVED / "tracking" / f"{SEQ}_{name}"
    ctc_tracks, ctc_masks = graph_to_ctc(graph, masks_tracked, outdir=str(outdir))
    tracked[name] = {"graph": graph, "masks": np.asarray(ctc_masks), "outdir": outdir}
    print(f"{name}: tracked in {time.time() - t0:.0f} s -> {outdir}")

/var/folders/zk/q7tvtcs93yg2rx1_q5rys65r0000gn/T/tmpdog1x8dq/general_2d.zip: 10
INFO:trackastra.model.model:Loading model state from /Users/inu/Library/Application Support/trackastra/models/general_2d/model.pt
INFO:trackastra.model.model_api:Using device mps
INFO:trackastra.model.model_api:Default batch size = 4 for model on mps.
INFO:trackastra.model.model_api:Predicting weights for candidate graph
INFO:trackastra.data.wrfeat:Extracting features from 92 frames.
INFO:trackastra.data.wrfeat:Using single process for feature extraction


Trackastra device: mps


Extracting features: 100%|█████████████████████| 92/92 [00:01<00:00, 65.09it/s]
INFO:trackastra.model.model_api:Building windows
Building windows: 100%|█████████████████████| 89/89 [00:00<00:00, 23964.37it/s]
INFO:trackastra.model.model_api:Predicting windows with batch size 4
Computing associations: 100%|██████████████████| 23/23 [00:04<00:00,  5.16it/s]
INFO:trackastra.model.model_api:Running greedy tracker
INFO:trackastra.tracking.tracking:Build candidate graph with delta_t=1
INFO:trackastra.tracking.tracking:Added 8307 vertices, 8180 edges              
INFO:trackastra.tracking.tracking:Running greedy tracker
Greedily matched edges: 100%|██████████| 8159/8159 [00:00<00:00, 362107.45it/s]
INFO:trackastra.model.model_api:Applying solution graph to masks
Saving masks: 100%|███████████████████████████| 92/92 [00:00<00:00, 140.26it/s]
INFO:trackastra.model.model_api:Predicting weights for candidate graph
INFO:trackastra.data.wrfeat:Extracting features from 92 frames.
INFO:trackastra.dat

finetuned: tracked in 7 s -> ../data/derived/tracking/01_finetuned


Extracting features: 100%|█████████████████████| 92/92 [00:01<00:00, 68.41it/s]
INFO:trackastra.model.model_api:Building windows
Building windows: 100%|█████████████████████| 89/89 [00:00<00:00, 24550.68it/s]
INFO:trackastra.model.model_api:Predicting windows with batch size 4
Computing associations: 100%|██████████████████| 23/23 [00:02<00:00,  8.24it/s]
INFO:trackastra.model.model_api:Running greedy tracker
INFO:trackastra.tracking.tracking:Build candidate graph with delta_t=1
INFO:trackastra.tracking.tracking:Added 8602 vertices, 8554 edges              
INFO:trackastra.tracking.tracking:Running greedy tracker
Greedily matched edges: 100%|██████████| 8530/8530 [00:00<00:00, 368322.90it/s]
INFO:trackastra.model.model_api:Applying solution graph to masks
Saving masks: 100%|███████████████████████████| 92/92 [00:00<00:00, 307.72it/s]

silvertruth: tracked in 5 s -> ../data/derived/tracking/01_silvertruth


## 5. Evaluate against the reference (traccuracy)
- **CTC measures** use the CTC matching rule: a prediction matches a reference marker if it covers more than half of it.
- **Division measures** use an IoU-based matching and a buffer of up to 2 frames (= 1 h).

In [6]:
from traccuracy import run_metrics
from traccuracy.loaders import load_ctc_data
from traccuracy.matchers import CTCMatcher, IOUMatcher
from traccuracy.metrics import CTCMetrics, DivisionMetrics

gt_dir = DATA / f"{SEQ}_GT" / "TRA"

def load_gt():
    # traccuracy annotates errors in place -> load a fresh copy of the reference for every evaluation
    return load_ctc_data(str(gt_dir), str(gt_dir / "man_track.txt"), name="GT")

def load_pred(name):
    outdir = tracked[name]["outdir"]
    return load_ctc_data(str(outdir), str(sorted(outdir.glob("*.txt"))[0]), name=name)

rows = []
for name in tracked:
    ctc, _ = run_metrics(gt_data=load_gt(), pred_data=load_pred(name),
                         matcher=CTCMatcher(), metrics=[CTCMetrics()])
    div, _ = run_metrics(gt_data=load_gt(), pred_data=load_pred(name),
                         matcher=IOUMatcher(iou_threshold=0.1, one_to_one=True),
                         metrics=[DivisionMetrics(max_frame_buffer=2)])
    r = ctc[0]["results"]
    row = {"masks": name, "DET": r["DET"], "LNK": r["LNK"], "TRA": r["TRA"]}
    for buf in [0, 1, 2]:
        dr = div[0]["results"][f"Frame Buffer {buf}"]
        row[f"div_F1_buf{buf}"] = dr["Division F1"]
        row[f"div_precision_buf{buf}"] = dr["Division Precision"]
        row[f"div_recall_buf{buf}"] = dr["Division Recall"]
    rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(RESULTS / f"tracking_metrics_seq{SEQ}.csv", index=False)
print(metrics_df.round(3).T.to_string())

Computing node attributes: 100%|██████████████| 92/92 [00:00<00:00, 293.16it/s]
INFO:traccuracy.loaders._ctc:Running CTC format checks
Computing node attributes: 100%|██████████████| 92/92 [00:00<00:00, 262.12it/s]
INFO:traccuracy.loaders._ctc:Running CTC format checks
Matching frames: 100%|████████████████████████| 92/92 [00:00<00:00, 974.29it/s]
INFO:traccuracy.matchers._base:Matched 8232 out of 8639 ground truth nodes.
INFO:traccuracy.matchers._base:Matched 8218 out of 8271 predicted nodes.
Computing node attributes: 100%|██████████████| 92/92 [00:00<00:00, 296.70it/s]
INFO:traccuracy.loaders._ctc:Running CTC format checks
Computing node attributes: 100%|██████████████| 92/92 [00:00<00:00, 272.26it/s]
INFO:traccuracy.loaders._ctc:Running CTC format checks
Matching frames: 100%|████████████████████████| 92/92 [00:00<00:00, 782.56it/s]
INFO:traccuracy.matchers._base:Matched 8203 out of 8639 ground truth nodes.
INFO:traccuracy.matchers._base:Matched 8203 out of 8271 predicted nodes.
Co

                            0            1
masks               finetuned  silvertruth
DET                     0.951        0.995
LNK                     0.942        0.992
TRA                      0.95        0.995
div_F1_buf0             0.789        0.882
div_precision_buf0      0.826        0.851
div_recall_buf0         0.755        0.915
div_F1_buf1             0.789        0.882
div_precision_buf1      0.826        0.851
div_recall_buf1         0.755        0.915
div_F1_buf2             0.789        0.882
div_precision_buf2      0.826        0.851
div_recall_buf2         0.755        0.915


## 6. Export tracks for R
One row per nucleus and frame: track ID, position (µm), area (µm²), parent track (lineage).
Speed is computed later in R from consecutive positions of the same track.

In [7]:
def read_ctc_tracks(path):
    """CTC track table: label, first frame, last frame, parent (0 = none)."""
    return pd.read_csv(path, sep=r"\s+", header=None, names=["track", "begin", "end", "parent"])

for name, d in tracked.items():
    lineage = read_ctc_tracks(sorted(d["outdir"].glob("*.txt"))[0])
    rows = []
    for t in range(d["masks"].shape[0]):
        p = regionprops_table(d["masks"][t], properties=["label", "centroid", "area"])
        rows.append(pd.DataFrame({"frame": t, "track": p["label"],
                                  "y_um": p["centroid-0"] * PIXEL_SIZE_UM,
                                  "x_um": p["centroid-1"] * PIXEL_SIZE_UM,
                                  "area_um2": p["area"] * PIXEL_SIZE_UM ** 2}))
    obs = pd.concat(rows, ignore_index=True).merge(lineage, on="track", how="left")
    obs["time_h"] = obs["frame"] * FRAME_INTERVAL_MIN / 60
    out = RESULTS / f"tracks_{name}_seq{SEQ}.csv"
    obs.to_csv(out, index=False)
    n_div = (lineage["parent"] > 0).sum() // 2
    print(f"{name}: {obs['track'].nunique()} tracks, {len(obs)} observations, ~{n_div} divisions -> {out}")

finetuned: 284 tracks, 8271 observations, ~86 divisions -> ../results/tracks_finetuned_seq01.csv
silvertruth: 269 tracks, 8596 observations, ~101 divisions -> ../results/tracks_silvertruth_seq01.csv


## 7. Inspect tracks in napari
Tracks are drawn as lines over time; a division shows as a branching track.
Look especially at divisions: are both daughters linked to the right mother?

In [8]:
%gui qt
import napari

d = tracked["finetuned"]
napari_tracks, napari_graph, _ = graph_to_napari_tracks(d["graph"])
lo, hi = np.percentile(raw, [1, 99.8])

viewer = napari.Viewer()
viewer.add_image(raw, name=f"raw {SEQ}", colormap="gray", contrast_limits=(lo, hi))
viewer.add_labels(d["masks"], name="tracked (fine-tuned)")
viewer.add_tracks(napari_tracks, graph=napari_graph, name="tracks", tail_length=10)

100%|█████████████████████████████████████| 284/284 [00:00<00:00, 68466.62it/s]
INFO:OpenGL.acceleratesupport:No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


<Tracks layer 'tracks' at 0x7123a47f50>

## Results (2026-11-03, sequence 01)
- Segmentation of 92 frames with the fine-tuned model: ~30 min on the M2 (cached).
- Trackastra (general_2d, greedy): 7 s on MPS.
- Fine-tuned masks: DET 0.951, LNK 0.942, TRA 0.950; divisions F1 0.79 (precision 0.83, recall 0.76).
- Silver-truth masks: DET 0.995, LNK 0.992, TRA 0.995; divisions F1 0.88 (precision 0.85, recall 0.92).
- Frame buffer 0–2 makes no difference: detected divisions are placed in the correct frame.
- Interpretation: linking itself is near challenge level; the gap comes mainly from segmentation.
  Divisions are the weak point (~1/4 missed, ~1/6 false) → motion analysis around mitosis uses the
  reference tracks; our pipeline serves as a robustness check.